# Build the combined supplementary tables workbook

Automatically discovers `TableS<number>.csv` files, writes each table to a named Excel sheet with a full title row, and validates the saved workbook.

In [ ]:
suppressPackageStartupMessages({
  library(data.table)
  library(openxlsx)
})

table_dir <- "."
output_path <- file.path(table_dir, "Supplementary_Tables.xlsx")

csv_paths <- list.files(
  table_dir,
  pattern = "^TableS[0-9]+[.]csv$",
  full.names = TRUE
)
stopifnot(
  length(csv_paths) > 0L,
  all(file.exists(csv_paths)),
  all(file.info(csv_paths)$size > 0)
)

table_manifest <- data.table(
  source_file = basename(csv_paths),
  source_path = csv_paths
)
table_manifest[, table_number := as.integer(sub(
  "^TableS([0-9]+)[.]csv$", "\\1", source_file
))]
table_manifest[, table_id := sub("[.]csv$", "", source_file)]
table_manifest[, sheet_name := paste0("Supplementary Table S", table_number)]
title_lookup <- c(
  TableS1 = "Supplementary Table S1: Cohort characteristics",
  TableS3 = "Supplementary Table S3: Quantified TMEM106B glycopeptiforms and their use in downstream analyses",
  TableS4 = "Supplementary Table S4: TMEM106B peptide-level pQTL of the coding variant T185S (rs3173615)",
  TableS2 = "Supplementary Table S2: Per-glycopeptiform summary of GlycoWAS, PWAS, and AD/pQTL colocalization",
  TableS5 = "Supplementary Table S5: APOD glycopeptiform trans-colocalization at the GRN locus with AD GWAS"
)
table_manifest[, table_title := unname(title_lookup[table_id])]
setorder(table_manifest, table_number)
stopifnot(
  !anyNA(table_manifest$table_number),
  !anyDuplicated(table_manifest$table_number),
  !anyDuplicated(table_manifest$table_id),
  !anyDuplicated(table_manifest$sheet_name),
  !anyNA(table_manifest$table_title),
  all(nchar(table_manifest$sheet_name) <= 31L)
)

tables <- setNames(
  lapply(table_manifest$source_path, fread, check.names = FALSE),
  table_manifest$table_id
)
table_notes <- setNames(vector('list', length(tables)), names(tables))
# A footer note is a final row whose first column starts with "Note:" and whose other columns are empty
# (Tables S2 and S3); it is written below the table instead of inside it.
for (tid in names(tables)) {
  note_rows <- grepl('^Note:', as.character(tables[[tid]][[1]]))
  if (!any(note_rows)) next
  stopifnot(
    sum(note_rows) == 1L,
    which(note_rows) == nrow(tables[[tid]]),
    all(vapply(tables[[tid]][note_rows, -1], function(x) {
      all(is.na(x) | trimws(as.character(x)) == '')
    }, logical(1)))
  )
  table_notes[[tid]] <- as.character(tables[[tid]][[1]][note_rows])
  tables[[tid]] <- tables[[tid]][!note_rows]
}
placeholder_headers <- vapply(tables, function(table_data) {
  identical(names(table_data), paste0("V", seq_len(ncol(table_data))))
}, logical(1))
stopifnot(!any(placeholder_headers))
table_manifest[, `:=`(
  rows = vapply(table_id, function(id) nrow(tables[[id]]), integer(1)),
  columns = vapply(table_id, function(id) ncol(tables[[id]]), integer(1))
)]
stopifnot(all(table_manifest$rows > 0L), all(table_manifest$columns > 0L))
table_manifest[]

In [ ]:
# Check: this notebook only aggregates already-built TableS*.csv files into one workbook,
# it does not itself load gpAnnotation. Still, scan every discovered CSV directly (not just
# the tables known to carry gpIDs) so a future TableS<n> can't silently reintroduce one of
# the 54 excluded sole-modification Trp C-mannosylation glycopeptiforms (mislabeled as
# glycation) without this notebook catching it.
w_sole_exclude_gpids <- readLines("../2_main_figures/Figure1/data/w_sole_exclude_gpids.txt")
stopifnot(length(w_sole_exclude_gpids) == 54)
qc_remove_gpids <- readLines("../2_main_figures/Figure1/data/qc_remove_gpids.txt")
excluded_gpids <- c(qc_remove_gpids, w_sole_exclude_gpids)
hits <- lapply(tables, function(tbl) intersect(unique(unlist(lapply(tbl, as.character))), excluded_gpids))
hits <- hits[lengths(hits) > 0]
stopifnot(length(hits) == 0)
cat("No affected gpIDs found across", length(tables), "assembled CSV(s):", paste(table_manifest$source_file, collapse = ", "), "\n")

In [ ]:
wb <- createWorkbook(creator = "glycoQTL paper")
title_style <- createStyle(
  fontSize = 14, textDecoration = "bold",
  valign = "center", wrapText = TRUE
)
note_style <- createStyle(
  fontSize = 10, textDecoration = 'italic', fontColour = '#404040',
  valign = 'top', wrapText = TRUE
)

for (table_index in seq_len(nrow(table_manifest))) {
  table_id <- table_manifest$table_id[table_index]
  sheet_name <- table_manifest$sheet_name[table_index]
  table_title <- table_manifest$table_title[table_index]
  table_data <- tables[[table_id]]
  addWorksheet(wb, sheet_name, gridLines = FALSE)
  writeData(wb, sheet_name, table_title, startRow = 1L, startCol = 1L)
  mergeCells(wb, sheet_name, cols = seq_len(ncol(table_data)), rows = 1L)
  addStyle(wb, sheet_name, title_style, rows = 1L, cols = 1L)
  setRowHeights(wb, sheet_name, rows = 1L, heights = 30)
  writeDataTable(
    wb, sheet_name, table_data,
    startRow = 3L,
    tableName = paste0("tbl_", table_id),
    tableStyle = "TableStyleMedium2",
    withFilter = TRUE
  )
  freezePane(wb, sheet_name, firstActiveRow = 4L)

  if (length(table_notes[[table_id]])) {
    note_row <- 3L + nrow(table_data) + 2L
    writeData(
      wb, sheet_name, table_notes[[table_id]],
      startRow = note_row, startCol = 1L
    )
    mergeCells(wb, sheet_name, cols = seq_len(ncol(table_data)), rows = note_row)
    addStyle(
      wb, sheet_name, note_style,
      rows = note_row, cols = seq_len(ncol(table_data)), gridExpand = TRUE
    )
    setRowHeights(wb, sheet_name, rows = note_row, heights = 32)
  }

  column_widths <- vapply(seq_along(table_data), function(column_index) {
    values <- as.character(table_data[[column_index]])
    max_width <- max(
      nchar(c(names(table_data)[column_index], values), type = "width"),
      na.rm = TRUE
    )
    min(60, max(10, max_width + 2))
  }, numeric(1))
  setColWidths(
    wb, sheet_name,
    cols = seq_along(table_data),
    widths = column_widths
  )
}

saveWorkbook(wb, output_path, overwrite = TRUE)
stopifnot(file.exists(output_path), file.info(output_path)$size > 0)
message("Saved: ", output_path)

In [ ]:
saved_sheets <- getSheetNames(output_path)
stopifnot(identical(saved_sheets, table_manifest$sheet_name))

validation <- rbindlist(lapply(seq_len(nrow(table_manifest)), function(table_index) {
  table_id <- table_manifest$table_id[table_index]
  this_sheet <- table_manifest$sheet_name[table_index]
  expected_title <- table_manifest$table_title[table_index]
  expected <- tables[[table_id]]
  observed_title <- as.character(read.xlsx(
    output_path, sheet = this_sheet, rows = 1L, cols = 1L,
    colNames = FALSE, skipEmptyRows = FALSE
  )[1, 1])
  observed <- as.data.table(read.xlsx(
    output_path,
    sheet = this_sheet,
    rows = 3L:(3L + nrow(expected)),
    check.names = FALSE,
    sep.names = " "
  ))
  expected_note <- table_notes[[table_id]]
  if (length(expected_note)) {
    note_row <- 3L + nrow(expected) + 2L
    observed_note <- as.character(read.xlsx(
      output_path, sheet = this_sheet, rows = note_row, cols = 1L,
      colNames = FALSE, skipEmptyRows = FALSE
    )[1, 1])
    stopifnot(identical(observed_note, expected_note))
  }
  stopifnot(
    identical(observed_title, expected_title),
    nrow(observed) == nrow(expected),
    ncol(observed) == ncol(expected),
    identical(names(observed), names(expected))
  )
  data.table(
    sheet = this_sheet,
    rows = nrow(observed),
    columns = ncol(observed),
    title = observed_title,
    source = table_manifest$source_file[table_index]
  )
}))

cat(
  "Validated", nrow(validation), "sheets; workbook size:",
  format(file.info(output_path)$size, big.mark = ","), "bytes\n"
)
validation[]